# Employee Performance Prediction

Employee Performance Prediction
Employee Performance Prediction
Problem Statement:
A rapidly growing organization operating across multiple regions in India wants to better understand the factors that influence employee performance. With a diverse workforce and multiple departments, leadership is looking for a data-driven approach to identify high performers and employees who may require additional support or training.

The organization has collected historical employee data containing demographic information, workplace characteristics, training records, and performance indicators. Your task is to analyze this data and build a machine learning model that can accurately predict the performance category of employees.

This is a problem where employees are categorized into different performance levels such as High Performer, Average Performer, and Low Performer.

The objective is to predict the Performance_Category for employees in the test dataset using the attributes provided in the training data.

- Predictions must be written to a CSV file along with Employee_ID, which is the unique identifier.
- The submission file should contain predictions for every record in the test dataset.
- The training dataset should only be used to build and validate your predictive model.

Description of Data Attributes:
• Employee_ID – Unique identifier for each employee

• Age – Age of the employee

• Gender – Gender of the employee

• City – City of the employee

• Department – Department where the employee works

• Education_Level – Highest educational qualification

• Experience_Years – Total years of professional experience

• Monthly_Salary – Monthly salary of the employee

• Projects_Completed – Number of projects completed by the employee

• Training_Hours – Total training hours attended

• Certification_Count – Number of professional certifications obtained

• Work_Life_Balance – Rating indicating employee’s work-life balance

• Job_Satisfaction – Satisfaction score reported by the employee

• Manager_Rating – Performance rating given by the reporting manager

• Overtime_Hours – No of hours that the employee worked overtime

• Commute_Time_Min – Travel time taken by the employee given in minutes

• Laptop_Issue_Count – No of times a laptop was issued to the employee

• Cafeteria_Rating – The rating that employee gave to the cafeteria

• Internet_Stability – Feedback about the stability of internet connection

• Last_Promotion_Years – Years since last promotion

• Absenteeism_Days – Number of absence days in the past year

• Performance_Category – Target variable indicating employee performance class

Evaluation Metric:
Accuracy Score

# **Setup & Import the libraries**

In [280]:
#Importing the necessary python libraries
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.2f' % x)

print("✅ Libraries imported successfully!")
print(f"Analysis Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

✅ Libraries imported successfully!
Analysis Date: 2026-03-23 13:56:01


# Load the dataset

We start by loading the transaction dataset and getting an overview of its structure, shape, and basic statistics.

In [281]:
train= pd.read_csv(r'/content/perf_train.csv-1773986173225')

print("DATASET OVERVIEW")

print(f"\nShape: {train.shape[0]:,} rows × {train.shape[1]} columns")
print(f"\nFirst 5 rows:")
print(train.head())
print(f"\nData Types:\n{train.dtypes}")
print(f"\nBasic Statistics:\n{train.describe()}")

DATASET OVERVIEW

Shape: 2,000 rows × 22 columns

First 5 rows:
   Employee_ID   Age  Gender       City Education_Level Department  \
0        10001 49.00    Male  Bengaluru   Post Graduate      Sales   
1        10002 35.00  Female  Bengaluru   Post Graduate  Analytics   
2        10003 28.00  Female        NaN        Graduate    Finance   
3        10004 41.00    Male  Hyderabad        Graduate      Sales   
4        10005 39.00    Male        NaN        Graduate      Sales   

   Experience_Years  Monthly_Salary  Projects_Completed  Training_Hours  \
0               NaN             NaN               15.00           20.00   
1              0.80       129555.00               15.00          163.00   
2             12.70        55760.00                 NaN          213.00   
3              4.90       128174.00                 NaN          209.00   
4              1.20       145192.00                5.00          117.00   

   Certifications  Work_Life_Balance  Job_Satisfaction  Manager_

As it is observed above that we have 2000 rows and 22 columns.
The first 21 columns represent the features and the last column (Performance_Category) represent the target/label.

# 3. Data Quality Assessment

In [282]:
from enum import unique
print("DATA QUALITY ASSESSMENT")
print(f"\n1. Missing Values:\n{train.isnull().sum()}")
print(f"\n2. Duplicate Transactions: {train.duplicated().sum()}")
print(f"\n3. Data Types:\n{train.dtypes}")
unique_employee    = train['Employee_ID'].nunique()
print(f"\n4. Unique Products: {unique_employee}")

DATA QUALITY ASSESSMENT

1. Missing Values:
Employee_ID               0
Age                     127
Gender                   66
City                    211
Education_Level         194
Department              199
Experience_Years        253
Monthly_Salary          105
Projects_Completed      288
Training_Hours          298
Certifications          305
Work_Life_Balance       181
Job_Satisfaction        196
Manager_Rating          147
Overtime_Hours          190
Commute_Time_Min        195
Laptop_Issue_Count       67
Cafeteria_Rating        283
Internet_Stability      343
Last_Promotion_Years    164
Absenteeism_Days        319
Performance_Category      0
dtype: int64

2. Duplicate Transactions: 0

3. Data Types:
Employee_ID               int64
Age                     float64
Gender                   object
City                     object
Education_Level          object
Department               object
Experience_Years        float64
Monthly_Salary          float64
Projects_Completed      f

In [283]:
##For complete information about the data and datatypes in the data
#train.info()

In [284]:
#To check if there are any special characters in place of values
#for i in data.columns:
    #print({i:df[i].unique()})

In [285]:
test = pd.read_csv("/content/perf_test.csv-1773986175060")
train.isnull().sum()

,0
Employee_ID,0
Age,127
Gender,66
City,211
Education_Level,194
Department,199
Experience_Years,253
Monthly_Salary,105
Projects_Completed,288
Training_Hours,298


#Data Cleaning

Remove Unnecessary Columns

In [286]:
# Drop ID (no predictive value)
#train.drop(columns=['Employee_ID'], inplace=True)
#test_ids = test['Employee_ID']
#test.drop(columns=['Employee_ID'], inplace=True)

In [287]:
# Consolidating all preprocessing steps for consistency

# RELOAD DATASETS TO ENSURE CLEAN STATE
train = pd.read_csv('/content/perf_train.csv-1773986173225')
test = pd.read_csv('/content/perf_test.csv-1773986175060')

print(f"Columns in train after reload: {train.columns.tolist()}")
print(f"Columns in test after reload: {test.columns.tolist()}")

# Store Employee_ID before dropping from test
test_ids = test['Employee_ID']

# Drop Employee_ID from both train and test
train.drop(columns=['Employee_ID'], inplace=True)
test.drop(columns=['Employee_ID'], inplace=True)

# Impute missing numerical and categorical values
# Re-identify num_cols and cat_cols as their content might have changed after dropping Employee_ID
num_cols = train.select_dtypes(include=['int64','float64']).columns.tolist()
cat_cols = train.select_dtypes(include=['object']).columns.tolist()
if 'Performance_Category' in cat_cols:
    cat_cols.remove('Performance_Category') # Target variable should not be imputed

# Numerical -> median
for col in num_cols:
    median = train[col].median()
    train[col].fillna(median, inplace=True)
    test[col].fillna(median, inplace=True)

# Categorical -> mode
for col in cat_cols:
    mode = train[col].mode()[0]
    train[col].fillna(mode, inplace=True)
    test[col].fillna(mode, inplace=True)

# Ordinal encoding (Education_Level, Internet_Stability)
edu_map = {'Graduate':1, 'Post Graduate':2, 'PhD':3}
train['Education_Level'] = train['Education_Level'].map(edu_map)
test['Education_Level'] = test['Education_Level'].map(edu_map)

internet_map = {'Poor':1, 'Average':2, 'Good':3, 'Excellent':4}
train['Internet_Stability'] = train['Internet_Stability'].map(internet_map)
test['Internet_Stability'] = test['Internet_Stability'].map(internet_map)

# Cap outliers for numerical columns
def cap_outliers(df, cols):
    for col in cols:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR
        df[col] = np.where(df[col] < lower, lower, np.where(df[col] > upper, upper, df[col]))
    return df

# Update num_cols to include newly mapped ordinal features if they became numerical
current_num_cols = train.select_dtypes(include=['int64','float64']).columns.tolist()
train = cap_outliers(train, current_num_cols)
test = cap_outliers(test, current_num_cols)

# Feature Engineering
def feature_engineering(df):
    df['Projects_per_Year'] = df['Projects_Completed'] / (df['Experience_Years'] + 1)
    df['Learning_Index'] = df['Training_Hours'] + df['Certifications']
    df['Engagement_Score'] = (df['Job_Satisfaction'] + df['Work_Life_Balance']) / 2
    df['Stress_Index'] = df['Overtime_Hours'] + df['Absenteeism_Days']
    return df
train = feature_engineering(train)
test = feature_engineering(test)

# Drop unnecessary columns
drop_cols = [
    'Cafeteria_Rating',
    'Laptop_Issue_Count',
    'Commute_Time_Min'
]
train.drop(columns=drop_cols, inplace=True)
test.drop(columns=drop_cols, inplace=True)

# One-hot encode remaining categorical features and align columns
# Re-identify categorical columns after all previous transformations
current_cat_cols = train.select_dtypes(include=['object']).columns.tolist()
if 'Performance_Category' in current_cat_cols:
    current_cat_cols.remove('Performance_Category')

train = pd.get_dummies(train, columns=current_cat_cols, drop_first=True)
test = pd.get_dummies(test, columns=current_cat_cols, drop_first=True)

# Align columns to ensure both have the same set of features in the same order
# fill_value=0 is important for new columns in test that might not be in train
train, test = train.align(test, join='left', axis=1, fill_value=0)

print("Preprocessing complete. Train and Test dataframes are now aligned.")

# Explicitly save the final processed dataframes to new variables
train_final = train.copy()
test_final = test.copy()

# Delete the original train and test to avoid confusion and ensure subsequent steps use train_final/test_final
del train
del test

Columns in train after reload: ['Employee_ID', 'Age', 'Gender', 'City', 'Education_Level', 'Department', 'Experience_Years', 'Monthly_Salary', 'Projects_Completed', 'Training_Hours', 'Certifications', 'Work_Life_Balance', 'Job_Satisfaction', 'Manager_Rating', 'Overtime_Hours', 'Commute_Time_Min', 'Laptop_Issue_Count', 'Cafeteria_Rating', 'Internet_Stability', 'Last_Promotion_Years', 'Absenteeism_Days', 'Performance_Category']
Columns in test after reload: ['Employee_ID', 'Age', 'Gender', 'City', 'Education_Level', 'Department', 'Experience_Years', 'Monthly_Salary', 'Projects_Completed', 'Training_Hours', 'Certifications', 'Work_Life_Balance', 'Job_Satisfaction', 'Manager_Rating', 'Overtime_Hours', 'Commute_Time_Min', 'Laptop_Issue_Count', 'Cafeteria_Rating', 'Internet_Stability', 'Last_Promotion_Years', 'Absenteeism_Days']
Preprocessing complete. Train and Test dataframes are now aligned.


In [288]:
#num_cols = train.select_dtypes(include=['int64','float64']).columns.tolist()
# The 'Performance_Category' column is of 'object' type and is not in num_cols, so this line is not needed.
# num_cols.remove('Performance_Category')

#for col in num_cols:
    #median_val = train[col].median()
    #train[col].fillna(median_val, inplace=True)
    #test[col].fillna(median_val, inplace=True)

In [289]:
##cat_cols = train.select_dtypes(include=['object']).columns.tolist()
#cat_cols.remove('Performance_Category')

#for col in cat_cols:
    ##mode_val = train[col].mode()[0]
    #train[col].fillna(mode_val, inplace=True)
    #test[col].fillna(mode_val, inplace=True)

In [290]:
# This cell is now empty as its logic has been moved to cell 3RgzoMmFwOOu for consolidation.

Salary & experience are skewed
Median is robust vs mean

Missing handled here
Later improved via Target Encoding

In [291]:
# This cell is now empty as its logic has been moved to cell 3RgzoMmFwOOu for consolidation.

Ordinal encoding preserves order relationships

In [292]:
# This cell is now empty as its logic has been moved to cell 3RgzoMmFwOOu for consolidation.

#Feature Engineering

In [293]:
#train['Experience_Age_Ratio'] = train['Experience_Years'] / (train['Age'] + 1)
#test['Experience_Age_Ratio'] = test['Experience_Years'] / (test['Age'] + 1)

Detects early achievers vs late starters

In [294]:
#train['Projects_per_Year'] = train['Projects_Completed'] / (train['Experience_Years'] + 1)
#test['Projects_per_Year'] = test['Projects_Completed'] / (test['Experience_Years'] + 1)

High performers → more output per year

In [295]:
#train['Learning_Index'] = train['Training_Hours'] + train['Certifications']
#test['Learning_Index'] = test['Training_Hours'] + test['Certifications']

Learning mindset = performance boost

In [296]:
#train['Engagement_Score'] = (
   # train['Job_Satisfaction'] + train['Work_Life_Balance']
#) / 2

#test['Engagement_Score'] = (
   # test['Job_Satisfaction'] + test['Work_Life_Balance']
#) / 2

Combines emotional + work balance signals

In [297]:
#train['Stress_Index'] = train['Overtime_Hours'] + train['Absenteeism_Days']
#test['Stress_Index'] = test['Overtime_Hours'] + test['Absenteeism_Days']

High stress → burnout → low performance

In [298]:
#train['Years_Since_Promotion_Ratio'] = train['Last_Promotion_Years'] / (train['Experience_Years'] + 1)
#test['Years_Since_Promotion_Ratio'] = test['Last_Promotion_Years'] / (test['Experience_Years'] + 1)

Long gap → stagnation → performance drop

In [299]:
#train['Salary_per_Experience'] = train['Monthly_Salary'] / (train['Experience_Years'] + 1)
#test['Salary_per_Experience'] = test['Monthly_Salary'] / (test['Experience_Years'] + 1)

Detects overpaid vs underpaid employees

In [300]:
#train['Work_Efficiency'] = (
    #train['Projects_Completed'] / (train['Overtime_Hours'] + 1)
#)

#test['Work_Efficiency'] = (
    #test['Projects_Completed'] / (test['Overtime_Hours'] + 1)
#)

In [301]:
# This cell is now empty as its logic has been moved to cell 3RgzoMmFwOOu for consolidation.

High output with low overtime = top performer

In [302]:
# This cell is now empty as its logic has been moved to cell 3RgzoMmFwOOu for consolidation.

#Feature Selection

Encode Target

In [303]:
# To calculate correlations, all columns must be numerical.
# The 'Performance_Category' was already encoded into 'y_encoded' during model setup.

# Create a DataFrame for correlation with numerical features and encoded target
features_for_corr = X.copy() # X already contains numerical features from train_final
features_for_corr['Performance_Category'] = y_encoded

corr = features_for_corr.corr()

# Check relation with target
corr_target = corr['Performance_Category'].sort_values(ascending=False)
print(corr_target)

Performance_Category     1.00
Projects_per_Year        0.18
Manager_Rating           0.12
City_Kolkata             0.06
Absenteeism_Days         0.03
Department_HR            0.03
Last_Promotion_Years     0.02
Job_Satisfaction         0.01
Department_Sales         0.01
Certifications           0.01
Learning_Index           0.01
Training_Hours           0.01
Stress_Index             0.00
Internet_Stability       0.00
Department_IT            0.00
City_Mumbai              0.00
Gender_Other            -0.00
Department_Operations   -0.00
Overtime_Hours          -0.01
City_Hyderabad          -0.01
Gender_Male             -0.01
City_Chennai            -0.01
Department_Finance      -0.01
City_Delhi              -0.02
Engagement_Score        -0.02
City_Pune               -0.02
Education_Level         -0.03
Age                     -0.03
Monthly_Salary          -0.04
Work_Life_Balance       -0.04
Experience_Years        -0.30
Projects_Completed      -0.38
Name: Performance_Category, dtype: float

In [304]:
from sklearn.feature_selection import mutual_info_classif

# # Identify all numerical columns in X, including engineered features
# current_num_cols = X.select_dtypes(include=['int64', 'float64']).columns

# # Fill any remaining NaNs in these numerical columns with their median
# for col in current_num_cols:
#     median_val = X[col].median()
#     X[col].fillna(median_val, inplace=True)

# X_temp = pd.get_dummies(X)
# mi = mutual_info_classif(X_temp, y)

# mi_scores = pd.Series(mi, index=X_temp.columns).sort_values(ascending=False)
# print(mi_scores.head(20)) # Commented out to prevent global X conflicts

In [305]:
# This cell is now empty as its logic has been moved to cell 3RgzoMmFwOOu for consolidation.

In [306]:
# from sklearn.preprocessing import LabelEncoder

# le = LabelEncoder()
# y = le.fit_transform(y)
# Commenting this out as LabelEncoder will be initialized and used in the model building cell for consistency.

In [307]:
from sklearn.ensemble import RandomForestClassifier

# rf = RandomForestClassifier(random_state=42)
# rf.fit(X_temp, y) # Commented out to prevent global X conflicts

# importance = pd.Series(rf.feature_importances_, index=X_temp.columns)
# importance.sort_values(ascending=False).head(20) # Commented out to prevent global X conflicts

In [308]:
print(X.dtypes)

Age                      float64
Education_Level          float64
Experience_Years         float64
Monthly_Salary           float64
Projects_Completed       float64
Training_Hours           float64
Certifications           float64
Work_Life_Balance        float64
Job_Satisfaction         float64
Manager_Rating           float64
Overtime_Hours           float64
Internet_Stability       float64
Last_Promotion_Years     float64
Absenteeism_Days         float64
Projects_per_Year        float64
Learning_Index           float64
Engagement_Score         float64
Stress_Index             float64
Gender_Male                 bool
Gender_Other                bool
City_Chennai                bool
City_Delhi                  bool
City_Hyderabad              bool
City_Kolkata                bool
City_Mumbai                 bool
City_Pune                   bool
Department_Finance          bool
Department_HR               bool
Department_IT               bool
Department_Operations       bool
Department

In [309]:
from imblearn.over_sampling import SMOTE

# smote = SMOTE(random_state=42)

# # Convert remaining categorical features in X to numerical using one-hot encoding
# X_encoded = pd.get_dummies(X) # Commented out to prevent global X conflicts

# X_resampled, y_resampled = smote.fit_resample(X_encoded, y) # Commented out to prevent global X conflicts

Model Building

Model 1: Random Forest

In [310]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Create the feature matrix X and target vector y from the fully preprocessed train_final DataFrame
X = train_final.drop('Performance_Category', axis=1)
y = train_final['Performance_Category']

# Apply Label Encoding to the target variable y
le = LabelEncoder()
y_encoded = le.fit_transform(y)

rf = RandomForestClassifier(random_state=42)
X_train, X_val, y_train, y_val = train_test_split(
    X, y_encoded, test_size=0.2, stratify=y_encoded, random_state=42
)

In [311]:
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, classification_report

param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [5, 10, None],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2]
}

grid = GridSearchCV(
    rf,
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

grid.fit(X_train, y_train)

best_rf = grid.best_estimator_

print("Best Parameters:", grid.best_params_)

Best Parameters: {'max_depth': None, 'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 200}


In [312]:
y_pred = best_rf.predict(X_val)

print("Accuracy:", accuracy_score(y_val, y_pred))
print("\nClassification Report:\n", classification_report(y_val, y_pred))

Accuracy: 0.885

Classification Report:
               precision    recall  f1-score   support

           0       0.85      0.99      0.92       226
           1       0.88      0.51      0.65        41
           2       0.96      0.82      0.88       133

    accuracy                           0.89       400
   macro avg       0.90      0.77      0.82       400
weighted avg       0.89      0.89      0.88       400



In [317]:
test_final_features = test_final.drop('Performance_Category', axis=1)
test_preds = best_rf.predict(test_final_features)
test_preds = le.inverse_transform(test_preds)

submission = pd.DataFrame({
    "Employee_ID": test_ids,
    "Performance_Category": test_preds
})

submission.to_csv("submission_rf.csv", index=False)